# 9장. 회귀 분석으로 숫자 예측하기

이 노트북은 `book/chapters/ch09_regression_analysis.md`와 `src/regression.py`의 회귀 분석 흐름을 그대로 실행합니다.

핵심은 높은 성능을 만드는 것이 아니라 다음 원칙을 지키는 것입니다.

- 예측 시점을 먼저 정의합니다.
- 주문 상세의 수량·단가·금액은 목표값 생성에만 사용합니다.
- 식별자와 예측 이후 정보를 입력값에서 제외합니다.
- 날짜 순서로 훈련·테스트 데이터를 나눕니다.
- 단순 평균 베이스라인보다 실제로 나은지 확인합니다.
- 낮거나 음수인 R²도 숨기지 않고 해석합니다.


## 0. 실행 전 확인

이 노트북은 5장에서 만든 다음 파일을 사용합니다.

- `data/processed/customers_clean.csv`
- `data/processed/orders_clean.csv`
- `data/processed/order_items_clean.csv`

파일이 없다면 프로젝트 루트에서 다음 명령을 먼저 실행합니다.

```bash
python scripts/preprocess_data.py
```


## 1. 회귀 문제와 예측 시점

예측 대상은 주문별 주문 상세 금액 합계인 `order_total`입니다.

교육용 예측 시점에서는 주문 시점과 결제 수단, 고객의 비식별 특성은 알 수 있지만 주문 상세의 수량·단가·금액은 모델에 제공하지 않는다고 가정합니다.

따라서 `quantity`, `unit_price`, `line_total`, `item_count`, `total_quantity`, `avg_unit_price`, `order_status`, `order_id`, `customer_id`는 입력값으로 사용하지 않습니다.


## 2. 패키지와 프로젝트 경로 설정


In [1]:
from pathlib import Path
import sys

import pandas as pd

CURRENT_DIR = Path.cwd()
PROJECT_ROOT = (
    CURRENT_DIR.parent
    if CURRENT_DIR.name == "notebooks"
    else CURRENT_DIR
)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
REPORT_DIR = PROJECT_ROOT / "reports"

from src.regression import (
    CATEGORICAL_FEATURES,
    FEATURE_COLUMNS,
    FORBIDDEN_FEATURES,
    NUMERIC_FEATURES,
    TARGET_COLUMN,
    build_leakage_checklist,
    build_regression_dataset,
    create_prediction_result,
    cross_validate_regression_models,
    load_regression_source_data,
    run_regression_analysis,
    save_regression_outputs,
    select_diagnostic_model,
    split_model_data_by_time,
    train_and_evaluate_models,
    validate_feature_columns,
)

print("프로젝트 루트:", PROJECT_ROOT.resolve())
print("전처리 데이터:", PROCESSED_DIR.resolve())
print("보고서 폴더:", REPORT_DIR.resolve())


프로젝트 루트: E:\git\llm-data-analysis-course
전처리 데이터: E:\git\llm-data-analysis-course\data\processed
보고서 폴더: E:\git\llm-data-analysis-course\reports


## 3. 전처리 데이터 불러오기

원본 CSV로 자동 대체하지 않습니다. 5장의 전처리 기준과 다른 데이터가 섞이면 실습 결과를 비교하기 어렵기 때문입니다.


In [2]:
data = load_regression_source_data(PROCESSED_DIR)

customers = data["customers"]
orders = data["orders"]
order_items = data["order_items"]

print("customers:", customers.shape)
print("orders:", orders.shape)
print("order_items:", order_items.shape)


customers: (150, 6)
orders: (300, 7)
order_items: (764, 6)


## 4. 누수 없는 주문 단위 모델링 데이터 만들기

`order_total`은 주문 상세에서 계산하지만, 주문 상세에서 만든 다른 집계값은 모델 입력에 포함하지 않습니다.


In [3]:
model_data = build_regression_dataset(
    customers=customers,
    orders=orders,
    order_items=order_items,
)

print("모델링 데이터:", model_data.shape)
display(
    model_data[
        [
            "order_id",
            "order_date",
            *FEATURE_COLUMNS,
            TARGET_COLUMN,
        ]
    ].head()
)


모델링 데이터: (300, 11)


,order_id,order_date,order_month,order_dayofweek,age,payment_method,gender,city,order_total
0,238,2025-09-15,9,0,67,bank_transfer,M,대구,361000
1,79,2025-09-17,9,2,63,bank_transfer,F,광주,283000
2,12,2025-09-18,9,3,61,kakao_pay,F,인천,444000
3,169,2025-09-18,9,3,69,kakao_pay,M,서울,448000
4,152,2025-09-19,9,4,63,naver_pay,F,광주,985000


In [4]:
print("숫자형 입력값:", NUMERIC_FEATURES)
print("범주형 입력값:", CATEGORICAL_FEATURES)
print("예측 대상:", TARGET_COLUMN)
print("금지 입력값:", sorted(FORBIDDEN_FEATURES))

validate_feature_columns(FEATURE_COLUMNS)


숫자형 입력값: ['order_month', 'order_dayofweek', 'age']
범주형 입력값: ['payment_method', 'gender', 'city']
예측 대상: order_total
금지 입력값: ['avg_unit_price', 'customer_id', 'item_count', 'line_total', 'order_id', 'order_status', 'order_total', 'quantity', 'total_quantity', 'unit_price']


## 5. 날짜 순서로 훈련·테스트 데이터 분할

과거 주문으로 이후 주문을 예측하는 상황을 모방하기 위해 무작위 분할 대신 날짜 순서 분할을 사용합니다.


In [5]:
train_data, test_data = split_model_data_by_time(
    model_data,
    test_size=0.2,
)

X_train = train_data[FEATURE_COLUMNS].copy()
X_test = test_data[FEATURE_COLUMNS].copy()
y_train = train_data[TARGET_COLUMN].copy()
y_test = test_data[TARGET_COLUMN].copy()

print(
    "훈련 기간:",
    train_data["order_date"].min(),
    "~",
    train_data["order_date"].max(),
)
print(
    "테스트 기간:",
    test_data["order_date"].min(),
    "~",
    test_data["order_date"].max(),
)
print("훈련:", X_train.shape, "테스트:", X_test.shape)


훈련 기간: 2025-09-15 00:00:00 ~ 2026-06-15 00:00:00
테스트 기간: 2026-06-15 00:00:00 ~ 2026-09-14 00:00:00
훈련: (240, 6) 테스트: (60, 6)


## 6. 베이스라인·선형 회귀·랜덤 포레스트 비교

각 모델의 전처리 파이프라인은 훈련 데이터 안에서만 결측치 대체, 표준화, 원-핫 인코딩을 학습합니다.

`Baseline Mean`은 훈련 목표값의 평균을 모든 테스트 주문에 예측합니다. 다른 모델은 최소한 이 기준과 비교해야 합니다.


In [6]:
models, model_comparison, predictions = (
    train_and_evaluate_models(
        X_train=X_train,
        X_test=X_test,
        y_train=y_train,
        y_test=y_test,
        random_state=42,
    )
)

model_comparison


,model,train_MAE,test_MAE,test_RMSE,test_R2,MAE_improvement_vs_baseline_pct
0,Baseline Mean,412811.979167,455895.833333,531015.673566,-0.002361,0.00
1,Random Forest,325428.186087,477919.253955,560536.388609,-0.116907,-4.83
2,Linear Regression,396273.659357,485756.269817,569459.154070,-0.152749,-6.55


`MAE_improvement_vs_baseline_pct`가 양수이면 베이스라인보다 평균 절대 오차가 줄었습니다. 음수이면 복잡한 모델이 단순 평균보다도 나쁘다는 뜻입니다.

훈련 MAE가 매우 낮고 테스트 MAE가 크면 과적합 가능성을 확인합니다.


## 7. 시간 순서 교차검증

한 번의 테스트 기간 결과만 믿지 않고, 훈련 기간 안에서 여러 시점으로 나누어 성능의 평균과 변동을 확인합니다.


In [7]:
cv_summary = cross_validate_regression_models(
    models=models,
    X_train=X_train,
    y_train=y_train,
    max_splits=5,
)

cv_summary


,model,cv_MAE_mean,cv_MAE_std,cv_R2_mean
0,Random Forest,427968.380438,1886.996038,-0.149259
1,Linear Regression,502305.310304,95206.584830,-0.608958


교차검증 MAE 표준편차가 크면 기간에 따라 성능이 불안정할 수 있습니다. R²가 반복적으로 음수라면 현재 입력값으로 평균 예측을 안정적으로 넘지 못할 가능성이 큽니다.


## 8. 실제값·예측값과 잔차 확인

테스트 MAE가 가장 낮은 비베이스라인 모델을 진단용으로 선택합니다. 선택된 모델이 베이스라인보다 낫다는 뜻은 아니므로 비교표를 함께 확인해야 합니다.


In [8]:
selected_model_name = select_diagnostic_model(
    model_comparison
)

prediction_result = create_prediction_result(
    test_data=test_data,
    y_test=y_test,
    y_pred=predictions[selected_model_name],
    model_name=selected_model_name,
)

print("진단 모델:", selected_model_name)
prediction_result.head(10)


진단 모델: Random Forest


,order_id,order_date,actual_order_total,predicted_order_total,residual,abs_error,model
0,51,2026-08-22,1877000,526514.079326,1.350486e+06,1.350486e+06,Random Forest
1,96,2026-07-18,1690000,658992.907423,1.031007e+06,1.031007e+06,Random Forest
2,21,2026-08-26,1767000,765853.314889,1.001147e+06,1.001147e+06,Random Forest
3,145,2026-08-03,1863000,914048.864375,9.489511e+05,9.489511e+05,Random Forest
4,15,2026-07-30,1677000,729995.379912,9.470046e+05,9.470046e+05,Random Forest
5,183,2026-07-30,1685000,770637.030265,9.143630e+05,9.143630e+05,Random Forest
6,282,2026-07-01,1566000,666853.599626,8.991464e+05,8.991464e+05,Random Forest
7,168,2026-08-18,10000,819086.850422,-8.090869e+05,8.090869e+05,Random Forest
8,248,2026-06-15,55000,862956.219305,-8.079562e+05,8.079562e+05,Random Forest
9,76,2026-08-21,1671000,867932.567458,8.030674e+05,8.030674e+05,Random Forest


In [9]:
prediction_result["abs_error"].describe()


count    6.000000e+01
mean     4.779193e+05
std      2.953783e+05
min      8.775302e+03
25%      2.429041e+05
50%      4.523267e+05
75%      6.812099e+05
max      1.350486e+06
Name: abs_error, dtype: float64

`residual = 실제값 - 예측값`입니다.

- 잔차가 양수이면 실제값을 낮게 예측했습니다.
- 잔차가 음수이면 실제값을 높게 예측했습니다.
- 큰 절대 오차가 특정 기간이나 금액대에 몰리는지 확인합니다.


## 9. 체크리스트와 결과 저장

예측 결과에는 주문 식별자가 포함되므로 `_internal` 파일은 외부 공개용으로 사용하지 않습니다.


In [10]:
regression_checklist = build_leakage_checklist()
regression_checklist


,check_item,status
0,예측 시점이 명확한가?,□
1,목표값과 목표값의 계산 재료를 입력에서 제외했는가?,□
2,예측 이후에 알 수 있는 정보를 사용하지 않았는가?,□
3,식별자를 일반 숫자 변수로 사용하지 않았는가?,□
4,전처리기가 훈련 데이터 안에서만 학습되는가?,□
5,시간 순서 또는 업무 목적에 맞는 분할을 사용했는가?,□
6,단순 베이스라인과 비교했는가?,□
7,테스트 데이터로 최종 성능을 평가했는가?,□
8,"MAE, RMSE, R²를 올바르게 해석했는가?",□
9,음수 R²와 낮은 성능을 숨기지 않았는가?,□


In [11]:
output_paths = save_regression_outputs(
    model_data=model_data,
    train_data=train_data,
    test_data=test_data,
    model_comparison=model_comparison,
    cv_summary=cv_summary,
    prediction_result=prediction_result,
    checklist=regression_checklist,
    selected_model_name=selected_model_name,
    report_dir=REPORT_DIR,
)

for name, path in output_paths.items():
    print(f"{name}: {path}")


model_data_internal: e:\git\llm-data-analysis-course\reports\ch09_regression_model_data_internal.csv
model_comparison: e:\git\llm-data-analysis-course\reports\ch09_regression_model_comparison.csv
cv_summary: e:\git\llm-data-analysis-course\reports\ch09_regression_cv_summary.csv
predictions_internal: e:\git\llm-data-analysis-course\reports\ch09_regression_predictions_internal.csv
checklist: e:\git\llm-data-analysis-course\reports\ch09_regression_checklist.csv
report: e:\git\llm-data-analysis-course\reports\ch09_regression_report.md
actual_vs_predicted_figure: e:\git\llm-data-analysis-course\reports\figures\ch09_actual_vs_predicted.png
residual_figure: e:\git\llm-data-analysis-course\reports\figures\ch09_residual_histogram.png


저장되는 주요 결과는 다음과 같습니다.

- 모델 비교표
- 시간 순서 교차검증 요약
- 내부용 모델링 데이터와 예측 오차
- 회귀 분석 체크리스트와 Markdown 보고서
- 실제값·예측값 산점도
- 잔차 히스토그램


## 10. 공통 모듈로 전체 파이프라인 다시 실행하기

앞의 단계는 `src/regression.py`의 `run_regression_analysis()`로 한 번에 재현할 수 있습니다.


In [12]:
regression_result = run_regression_analysis(
    processed_dir=PROCESSED_DIR,
    report_dir=REPORT_DIR,
    test_size=0.2,
    random_state=42,
)

regression_result["model_comparison"]


,model,train_MAE,test_MAE,test_RMSE,test_R2,MAE_improvement_vs_baseline_pct
0,Baseline Mean,412811.979167,455895.833333,531015.673566,-0.002361,0.00
1,Random Forest,325428.186087,477919.253955,560536.388609,-0.116907,-4.83
2,Linear Regression,396273.659357,485756.269817,569459.154070,-0.152749,-6.55


## 11. 실행 스크립트 사용

프로젝트 루트에서 다음 명령을 실행하면 같은 파이프라인이 수행됩니다.

```bash
python scripts/run_regression_analysis.py
```


## 12. LLM에게 회귀 코드를 요청하는 프롬프트

```text
온라인 쇼핑몰 주문 금액을 예측하는 교육용 회귀 모델을 만들려고 합니다.

예측 대상:
- order_total: 주문별 line_total 합계

예측 시점:
- 주문 메타데이터와 고객의 비식별 특성은 알 수 있지만
  주문 상세의 수량, 단가, 금액은 모델에 제공하지 않음

사용 가능한 입력값:
- payment_method
- order_month
- order_dayofweek
- gender
- age
- city

사용하면 안 되는 입력값:
- order_total, line_total
- quantity, unit_price
- item_count, total_quantity, avg_unit_price
- order_status
- order_id, customer_id

요청:
1. 날짜 순서로 훈련 데이터 80%, 테스트 데이터 20%를 나누어 주세요.
2. 결측치 처리와 OneHotEncoder를 Pipeline 안에 넣어 주세요.
3. DummyRegressor, LinearRegression, RandomForestRegressor를 비교해 주세요.
4. MAE, RMSE, R²와 베이스라인 대비 개선율을 계산해 주세요.
5. 훈련·테스트 성능 차이와 데이터 누수 가능성을 설명해 주세요.
6. R²가 음수일 때의 의미도 설명해 주세요.

주의:
- 실제 데이터에 없는 컬럼을 만들지 마세요.
- 테스트 데이터로 전처리 규칙을 학습하지 마세요.
- 높은 성능을 가정하거나 결과를 임의로 만들어내지 마세요.
```


## 13. 실습 과제

1. `payment_method`를 제외했을 때 베이스라인 대비 성능이 어떻게 달라지는지 비교합니다.
2. `RandomForestRegressor`의 `min_samples_leaf` 값을 바꾸어 훈련·테스트 MAE를 비교합니다.
3. 시간 순서 교차검증에서 성능이 가장 낮은 기간의 데이터 특성을 확인합니다.
4. 예측 시점 이전의 고객 구매 이력을 설계하되 미래 정보가 섞이지 않도록 집계 기준일을 명시합니다.
5. LLM이 제안한 회귀 코드에서 목표값의 계산 재료나 식별자가 입력에 포함됐는지 검토합니다.


In [13]:
# 과제 1. FEATURE_COLUMNS에서 payment_method를 제외한 별도 실험을 설계해 보세요.


In [14]:
# 과제 2. RandomForestRegressor의 min_samples_leaf 값을 바꾸어 비교해 보세요.


## 14. 정리

이번 장에서는 다음 내용을 실습했습니다.

- 예측 시점 정의
- 목표값과 목표값 계산 재료 분리
- 식별자와 예측 이후 정보 제외
- 날짜 순서 훈련·테스트 분할
- Pipeline 안에서 결측치 처리와 인코딩
- 평균 베이스라인, 선형 회귀, 랜덤 포레스트 비교
- MAE, RMSE, R²와 베이스라인 대비 개선율 해석
- 시간 순서 교차검증
- 실제값·예측값과 잔차 진단
- 내부 식별 정보 보호
- `src/regression.py`와 실행 스크립트를 통한 재현

낮은 성능도 현재 데이터와 입력 변수의 한계를 알려 주는 유효한 분석 결과입니다.


# Chapter 09 답안 양식. 회귀 분석으로 숫자 예측하기

> 이 내용을 `chapter09.ipynb`의 Markdown 셀로 작성합니다.

## 제출 정보
- 이름:김혜빈
- GitHub ID:Chloe-Hyebin-Kim
- 작성일:2026-11-11
- 최종 Notebook URL: https://github.com/Chloe-Hyebin-Kim/llm-data-analysis-course/blob/main/practice/chapter09/chapter09.ipynb


## 1. 예측 문제 정의
- Target:
- 예측 단위:
- 예측 시점:
- 예측 시점에 알 수 있는 정보:
- 예측 시점에 알 수 없는 정보:

### 나의 해석과 판단
왜 이 예측 시점을 선택했는지 작성하세요.

---

## 2. Feature Leakage Audit

| feature | 예측 시점 사용 가능 | Target 계산 재료 | 사후 정보/ID | 최종 사용 | 판단 이유 |
| --- | --- | --- | --- | --- | --- |
|  |  |  |  |  |  |

- 금지 feature overlap 개수:
- 가장 위험하다고 판단한 feature:
- 이유:

![Leakage 점검](images/step02_leakage.png)

---

## 3. Target 생성과 관계 검증

### 계산 기준

```text
line_total =
order_total =
```

- line_total 검증 결과:
- orders ↔ target 관계 검증 결과:
- orders → customers 관계 검증 결과:

### 나의 해석과 판단
관계 검증 없이 병합만 수행하면 어떤 문제가 생길 수 있는지 작성하세요.

---

## 4. Train / Final Test 분할

- Train 시작일:
- Train 종료일:
- Train 행 수:
- Final Test 시작일:
- Final Test 종료일:
- Final Test 행 수:
- 같은 달력 날짜 중복 여부:

### 분할 판단
왜 이번 문제에서 시간 순서 분할을 사용하는지 작성하세요.

![시간 분할](images/step04_split.png)

---

## 5. Train TimeSeriesSplit 후보 비교

| 모델 | CV MAE 평균 | CV MAE 표준편차 | CV R² 평균 | 비고 |
| --- | ---: | ---: | ---: | --- |
| Baseline Mean | | | | |
| Linear Regression | | | | |
| Random Forest | | | | |

### Train CV로 선택한 비베이스라인 모델
- Selected Model:
- 선택 기준:
- Final Test 결과를 보기 전에 선택했는가: 예 / 아니오

### 나의 해석과 판단
CV 평균뿐 아니라 표준편차도 함께 봐야 하는 이유를 작성하세요.

---

## 6. Final Test: Baseline vs Frozen Model

| 모델 | selection role | Train MAE | Test MAE | Test RMSE | Test R² | Baseline 대비 MAE 개선율 |
| --- | --- | ---: | ---: | ---: | ---: | ---: |
| Baseline Mean | baseline | | | | | |
| Frozen Selected Model | selected_by_train_cv | | | | | |

![최종 회귀 평가](images/step06_final_metrics.png)

### 결과 관찰

### 나의 해석과 판단
- Frozen Model이 baseline보다 실제로 개선되었는가?
- MAE와 RMSE 차이에서 무엇을 볼 수 있는가?
- R²는 어떻게 해석할 수 있는가?

### 업무·분석적 의미
현재 오차 수준이 실제 사용에서 어떤 의미인지 작성하세요.

---

## 7. 대표 오차 사례

### 사례 1
- 실제값:
- 예측값:
- residual:
- absolute error:
- 관찰:
- 가능한 이유 후보:
- 추가 확인:

### 사례 2
- 실제값:
- 예측값:
- residual:
- absolute error:
- 관찰:
- 가능한 이유 후보:
- 추가 확인:

> 원인을 데이터가 직접 보여 주지 않는다면 사실처럼 단정하지 않습니다.

---

## 8. Validation Evidence

`reports/ch09_regression_validation.csv` 기준으로 작성하세요.

| check | value | status |
| --- | --- | --- |
| forbidden_feature_overlap | | |
| strict_train_before_test | | |
| selected_model_exists_in_train_cv | | |
| final_test_contains_baseline | | |
| final_test_contains_frozen_selected_model | | |
| test_rows_for_r2 | | |

### FAIL 항목
- 없음 / 있음:
- 있다면 원인과 수정 내용:

---

## 9. Evidence와 공개 범위

### 생성된 Evidence
- [ ] `ch09_regression_split_summary.csv`
- [ ] `ch09_regression_feature_audit.csv`
- [ ] `ch09_regression_cv_summary.csv`
- [ ] `ch09_regression_model_comparison.csv`
- [ ] `ch09_regression_validation.csv`
- [ ] `ch09_regression_checklist.csv`

### Internal 파일
- [ ] `ch09_regression_model_data_internal.csv`
- [ ] `ch09_regression_predictions_internal.csv`

### 공개 결과
- [ ] `ch09_regression_report.md`
- [ ] `ch09_actual_vs_predicted.png`
- [ ] `ch09_residual_histogram.png`

### 공개 범위 판단
왜 `order_id`가 포함된 예측 결과를 Internal로 구분해야 하는지 작성하세요.

---

## 10. 최종 사용 판단

- [ ] 현재 수준에서 참고용 사용 가능
- [ ] 추가 검증 후 사용 가능
- [ ] 현재 데이터로는 사용 보류

### 판단 근거
1.
2.
3.

### 업무적 위험

### 현재 데이터의 한계

### 다음 개선 우선순위
1.
2.
3.

---

## 11. 전체 재실행 확인

실행 명령:

```powershell
python scripts/run_regression_analysis.py
```

- 실행 성공 여부:
- Train CV Selected Model:
- Final Validation 전체 PASS 여부:
- Notebook 결과와 스크립트 결과의 일관성:

---

## 최종 체크

- [ ] Target과 예측 시점을 정의했습니다.
- [ ] Leakage Audit을 수행했습니다.
- [ ] Target 계산과 관계 검증을 확인했습니다.
- [ ] Train과 Final Test가 시간 순서로 엄격히 분리되었습니다.
- [ ] 전처리가 Pipeline 내부에서 학습됩니다.
- [ ] Train TimeSeriesSplit으로 후보를 비교했습니다.
- [ ] Final Test 전에 Selected Model을 고정했습니다.
- [ ] Final Test에서는 Baseline과 Frozen Model만 비교했습니다.
- [ ] MAE, RMSE, R²를 함께 해석했습니다.
- [ ] 대표 오차 사례를 관찰과 가설로 구분했습니다.
- [ ] Validation Evidence가 모두 PASS입니다.
- [ ] Internal 결과와 공개 결과를 구분했습니다.
- [ ] 낮은 성능도 숨기지 않았습니다.
- [ ] 전체 스크립트를 재실행했습니다.
- [ ] 최종 Notebook URL을 제출합니다.
